# Juliet Utility Router 학습

Frozen split을 유지한 채 전체 train/dev case를 materialize하고, Semantic Analyzer candidate를 캐시한 뒤 `5 Experts × LLM models` outcome matrix를 수집하여 Utility Router와 escalation gate를 학습합니다. `*_CASE_LIMIT = 0`은 해당 split 전체를 의미합니다. 유료 호출은 두 개의 잠금과 실행별 예산 제한을 통과해야 합니다.

In [ ]:
from pathlib import Path

EVAL_ROOT = Path.cwd().resolve()
if EVAL_ROOT.name != 'Model_Evaluation':
    EVAL_ROOT = (EVAL_ROOT / 'Model_Evaluation').resolve()
CONFIG_PATH = EVAL_ROOT / 'configs' / 'full.toml'
ENV_FILE = EVAL_ROOT.parent / '.env'
TRAIN_CASE_LIMIT = 0  # 0 = frozen train split 전체
DEV_CASE_LIMIT = 0    # 0 = frozen dev split 전체
MODEL_IDS = []        # 비우면 .env의 sweep/expert model을 사용
MAX_CANDIDATES_PER_CASE = 4
HARD_NEGATIVES_PER_CASE = 1
TARGET_TRUTH_RECALL = 0.95
EXECUTE_PAID = False  # 반드시 dry-run 확인 후 True
MAX_REQUESTS_PER_RUN = 500
MAX_USD_PER_RUN = 20.0
RESERVE_USD_PER_REQUEST = 0.10


In [ ]:
import json, sys
sys.path.insert(0, str(EVAL_ROOT / 'src'))
from model_evaluation.config import load_config, load_mapping
from model_evaluation.stages.materialize_dataset import materialize_dataset
from model_evaluation.candidates import cache_candidates
from model_evaluation.workflow import (resolve_models, plan_outcome_matrix, collect_outcome_matrix, audit_outcome_matrix, train_utility_router)
from model_evaluation.adapters.llm_security import expert_assignments
config = load_config(CONFIG_PATH)
mapping = load_mapping(config.paths.mapping)
RUN_DIR = EVAL_ROOT / 'work' / 'router_training'
RESULT_DIR = EVAL_ROOT / 'results' / 'router_training'
ARTIFACT = EVAL_ROOT / 'artifacts' / 'juliet_utility_router.pkl'
RUN_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)


## 1. Frozen train/dev case 생성

In [ ]:
materialization = materialize_dataset(
    config, mapping, output_directory=RUN_DIR / 'cases',
    splits=('train', 'dev'), limits={'train': TRAIN_CASE_LIMIT, 'dev': DEV_CASE_LIMIT},
    progress=print,
)
print(json.dumps(materialization, ensure_ascii=False, indent=2))

## 2. Semantic Analyzer candidate 캐시

In [ ]:
for split in ('train', 'dev'):
    summary = cache_candidates(
        RUN_DIR / 'cases' / f'cases_{split}.jsonl',
        RUN_DIR / 'candidates' / f'candidates_{split}.jsonl',
        max_source_bytes=config.max_source_bytes, parse_timeout_ms=config.parse_timeout_ms, progress=print,
    )
    print(split, json.dumps(summary, ensure_ascii=False, indent=2))

## 3. API dry-run: 모델/요청 수/재개 상태 확인

In [ ]:
models = resolve_models(ENV_FILE, MODEL_IDS)
plans = {}
for split in ('train', 'dev'):
    plans[split] = plan_outcome_matrix(
        cases_path=RUN_DIR / 'cases' / f'cases_{split}.jsonl',
        candidate_cache=RUN_DIR / 'candidates' / f'candidates_{split}.jsonl',
        selection_manifest=RUN_DIR / 'selections' / f'selected_{split}.jsonl',
        outcome_path=RUN_DIR / 'outcomes' / f'outcomes_{split}.jsonl',
        model_ids=models, max_candidates_per_case=MAX_CANDIDATES_PER_CASE,
        hard_negatives_per_case=HARD_NEGATIVES_PER_CASE,
    )
print(json.dumps(plans, ensure_ascii=False, indent=2))
print('주의: expected_api_requests는 논리적 Expert 호출 수입니다. 비용 상한은 실행별로 적용됩니다.')

## 4. Expert×Model outcome 수집

실행하려면 이 노트북의 `EXECUTE_PAID=True`와 부모 `.env`의 `RUN_PAID_EXPERIMENTS=1`이 모두 필요합니다. 상한에 걸리면 저장된 행부터 다음 실행에서 재개됩니다.

In [ ]:
if EXECUTE_PAID:
    for split in ('train', 'dev'):
        run = collect_outcome_matrix(
            env_file=ENV_FILE, cases_path=RUN_DIR / 'cases' / f'cases_{split}.jsonl',
            candidate_cache=RUN_DIR / 'candidates' / f'candidates_{split}.jsonl',
            outcome_path=RUN_DIR / 'outcomes' / f'outcomes_{split}.jsonl',
            ledger_path=RUN_DIR / 'ledgers' / f'{split}_api_ledger.jsonl', model_ids=models,
            execute_paid=True, max_requests=MAX_REQUESTS_PER_RUN, max_usd=MAX_USD_PER_RUN,
            reserve_usd_per_request=RESERVE_USD_PER_REQUEST,
            max_candidates_per_case=MAX_CANDIDATES_PER_CASE, hard_negatives_per_case=HARD_NEGATIVES_PER_CASE,
        )
        print(split, json.dumps(run, ensure_ascii=False, indent=2))
else:
    print('PAID LOCKED: 위 dry-run을 확인한 뒤에만 EXECUTE_PAID=True로 변경하세요.')

## 5. 완전성 검사 후 Utility Router + Escalation Gate 학습

In [ ]:
expected_ids = [item.assignment_id for item in expert_assignments(models)]
outcome_files = {s: RUN_DIR / 'outcomes' / f'outcomes_{s}.jsonl' for s in ('train', 'dev')}
audits = {s: audit_outcome_matrix(p, expected_assignment_ids=expected_ids, selection_manifest=RUN_DIR / 'selections' / f'selected_{s}.jsonl') if p.exists() else {'complete': False, 'reason': 'missing'} for s, p in outcome_files.items()}
print(json.dumps(audits, ensure_ascii=False, indent=2))
if all(item['complete'] for item in audits.values()):
    training_report = train_utility_router(
        train_outcomes=outcome_files['train'], dev_outcomes=outcome_files['dev'],
        train_selection_manifest=RUN_DIR / 'selections' / 'selected_train.jsonl',
        dev_selection_manifest=RUN_DIR / 'selections' / 'selected_dev.jsonl',
        artifact_path=ARTIFACT, report_path=RESULT_DIR / 'training_report.json',
        model_ids=models, seed=config.seed, target_truth_recall=TARGET_TRUTH_RECALL,
    )
    print(json.dumps(training_report, ensure_ascii=False, indent=2))
else:
    print('학습 보류: outcome 행렬 수집을 재개하여 train/dev를 모두 complete로 만드세요.')